# Instructor Setup: I-Beam Lab

Run this notebook before the lab. It generates a Latin Hypercube Sample of beams
for testing, gives you a data-entry template, and calibrates noise from duplicate tests.
The output is a CSV your students will load.

In [ ]:
!pip install -q scikit-learn scipy

import numpy as np
import pandas as pd
from scipy.stats.qmc import LatinHypercube
import matplotlib.pyplot as plt

# === EDIT THESE ===
B_BOUNDS = (1.0, 8.0)    # web thickness b (mm)
H_BOUNDS = (12.0, 23.0)  # web height H (mm)
N_BEAMS = 12             # initial LHS count
N_EXTRA = 4              # maximin infill points (0 to skip)
N_DUPLICATES = 2         # how many beams to retest for noise calibration
SEED = 42

In [ ]:
# Generate initial LHS
sampler = LatinHypercube(d=2, seed=SEED)
u = sampler.random(n=N_BEAMS)
b_init = B_BOUNDS[0] + u[:, 0] * (B_BOUNDS[1] - B_BOUNDS[0])
H_init = H_BOUNDS[0] + u[:, 1] * (H_BOUNDS[1] - H_BOUNDS[0])

X_init = np.column_stack([b_init, H_init])

# Maximin infill
if N_EXTRA > 0:
    pool_sampler = LatinHypercube(d=2, seed=SEED + 1000)
    u_pool = pool_sampler.random(n=500)
    b_pool = B_BOUNDS[0] + u_pool[:, 0] * (B_BOUNDS[1] - B_BOUNDS[0])
    H_pool = H_BOUNDS[0] + u_pool[:, 1] * (H_BOUNDS[1] - H_BOUNDS[0])
    pool = np.column_stack([b_pool, H_pool])

    # Normalize for distance calc
    def norm_pt(X):
        Xn = X.copy()
        Xn[:, 0] = (Xn[:, 0] - B_BOUNDS[0]) / (B_BOUNDS[1] - B_BOUNDS[0])
        Xn[:, 1] = (Xn[:, 1] - H_BOUNDS[0]) / (H_BOUNDS[1] - H_BOUNDS[0])
        return Xn

    chosen = []
    used = X_init.copy()
    for _ in range(N_EXTRA):
        used_n = norm_pt(used)
        pool_n = norm_pt(pool)
        min_dists = np.array([np.min(np.linalg.norm(pool_n[i] - used_n, axis=1)) for i in range(len(pool))])
        best = np.argmax(min_dists)
        chosen.append(pool[best])
        used = np.vstack([used, pool[best:best+1]])
        pool = np.delete(pool, best, axis=0)

    X_extra = np.array(chosen)
    X_all = np.vstack([X_init, X_extra])
else:
    X_extra = np.zeros((0, 2))
    X_all = X_init

print(f'Generated {len(X_init)} initial + {len(X_extra)} infill = {len(X_all)} beams')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_init[:, 0], X_init[:, 1], s=80, c='C0', label=f'Initial {len(X_init)}', zorder=5)
if len(X_extra) > 0:
    ax.scatter(X_extra[:, 0], X_extra[:, 1], s=80, c='C1', marker='s', label=f'Infill {len(X_extra)}', zorder=5)
ax.set_xlabel('Web thickness b (mm)')
ax.set_ylabel('Web height H (mm)')
ax.set_title('LHS Beam Designs')
ax.legend()
ax.set_xlim(B_BOUNDS)
ax.set_ylim(H_BOUNDS)
plt.tight_layout()
plt.show()

# Print-ready table
df = pd.DataFrame({'Beam': range(1, len(X_all)+1),
                    'b_web_mm': np.round(X_all[:, 0], 3),
                    'H_web_mm': np.round(X_all[:, 1], 3)})
print('\nBeams to print and test:')
print(df.to_string(index=False))

## Data Entry

After testing, fill in the `Str_w` column below with your measured strength-to-weight
ratio (N/g) for each beam. Pick N_DUPLICATES beams to retest and add those as extra rows
at the bottom with `is_duplicate = True`.

In [ ]:
# Fill in Str_w after testing. Add duplicate rows at the bottom.
# Example with placeholder values (replace -1 with your measurements):

data = {
    'Beam': list(range(1, len(X_all)+1)),
    'b_web_mm': list(np.round(X_all[:, 0], 3)),
    'H_web_mm': list(np.round(X_all[:, 1], 3)),
    'Str_w': [-1.0] * len(X_all),          # <-- REPLACE with measured values
    'is_duplicate': [False] * len(X_all),
}

# Add duplicate tests here. Example: retesting beams 3 and 7
# data['Beam'].extend([3, 7])
# data['b_web_mm'].extend([X_all[2, 0], X_all[6, 0]])
# data['H_web_mm'].extend([X_all[2, 1], X_all[6, 1]])
# data['Str_w'].extend([28.5, 30.1])  # <-- second measurement
# data['is_duplicate'].extend([True, True])

df_results = pd.DataFrame(data)
df_results

In [ ]:
# Noise calibration from duplicate tests
dupes = df_results[df_results['is_duplicate'] == True]
if len(dupes) == 0:
    print('No duplicates entered yet. Add duplicate rows above and rerun.')
    NOISE_BASELINE = 1e-3  # placeholder
else:
    # For each duplicated beam, compute variance in log-space
    variances = []
    for beam_id in dupes['Beam'].unique():
        all_measurements = df_results[df_results['Beam'] == beam_id]['Str_w'].values
        all_measurements = all_measurements[all_measurements > 0]
        if len(all_measurements) >= 2:
            log_vals = np.log(all_measurements)
            variances.append(np.var(log_vals, ddof=1))
    if variances:
        NOISE_BASELINE = np.mean(variances)
    else:
        NOISE_BASELINE = 1e-3
        print('Could not compute variance. Using placeholder.')

NOISE_HI = NOISE_BASELINE * 100
print(f'Noise baseline (log-space variance): {NOISE_BASELINE:.2e}')
print(f'Student noise bounds: [{NOISE_BASELINE:.2e}, {NOISE_HI:.2e}]')

In [ ]:
# Export CSV for students
# Drop duplicates for the main dataset (keep originals only)
df_export = df_results[df_results['is_duplicate'] == False][['Beam', 'b_web_mm', 'H_web_mm', 'Str_w']].copy()
df_export = df_export[df_export['Str_w'] > 0]  # only export rows with real data

out_path = 'beam_lab_data.csv'
df_export.to_csv(out_path, index=False)
print(f'Saved {len(df_export)} beams to {out_path}')
print(f'Upload this CSV to your GitHub repo data/ folder.')
print(f'Then update the DATA_URL in the student notebook.')
print()
print(f'Tell students their noise bounds are: [{NOISE_BASELINE:.2e}, {NOISE_HI:.2e}]')